In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

# load all three BCSC files
print("Loading BCSC data...")
df1 = pd.read_csv('../data/bcsc_risk_factors_summarized1_092020.csv')
df2 = pd.read_csv('../data/bcsc_risk_factors_summarized2_092020.csv')
df3 = pd.read_csv('../data/bcsc_risk_factors_summarized3_092020.csv')
df_raw = pd.concat([df1, df2, df3], ignore_index=True)

print(f"Raw shape: {df_raw.shape}")
print(f"Total patients represented: {df_raw['count'].sum():,}")

# ── decode BCSC variable coding ───────────────────────────
# age_group_5_years: 1=18-24, 2=25-29, ..., 13=78-84
# map to approximate midpoint age
age_map = {1:21, 2:27, 3:32, 4:37, 5:42, 6:47,
           7:52, 8:57, 9:62, 10:67, 11:72, 12:77, 13:82}

# age_menarche: 0=<12, 1=12, 2=13+, 9=unknown
menarche_map = {0:11, 1:12, 2:14, 9:np.nan}

# age_first_birth: 0=nulliparous, 1=<20, 2=20-24, 3=25-29, 4=30+, 9=unknown
first_birth_map = {0:0, 1:18, 2:22, 3:27, 4:32, 9:np.nan}

# bmi_group: 1=<18.5, 2=18.5-24.9, 3=25-29.9, 4=30+, 9=unknown
bmi_map = {1:17, 2:22, 3:27, 4:33, 9:np.nan}

# menopaus: 1=premenopausal, 2=postmenopausal, 3=surgical, 9=unknown
# current_hrt: 0=no, 1=yes, 9=unknown
# first_degree_hx: 0=no, 1=yes, 9=unknown
# biophx: 0=no, 1=yes, 9=unknown
# breast_cancer_history: 0=no, 1=yes, 9=unknown
# BIRADS_breast_density: 1=almost entirely fat, 2=scattered, 3=heterogeneous, 4=extremely dense

df = df_raw.copy()

# apply mappings
df['age']          = df['age_group_5_years'].map(age_map)
df['age_menarche'] = df['age_menarche'].map(menarche_map)
df['age_first_birth'] = df['age_first_birth'].map(first_birth_map)
df['bmi']          = df['bmi_group'].map(bmi_map)

# recode unknowns as NaN
for col in ['first_degree_hx', 'current_hrt', 'biophx', 'breast_cancer_history']:
    df[col] = df[col].replace(9, np.nan)

df['menopaus'] = df['menopaus'].replace(9, np.nan)
df['BIRADS_breast_density'] = df['BIRADS_breast_density'].replace(9, np.nan)

# target: breast cancer history
# drop unknown outcomes
df = df[df['breast_cancer_history'].notna()].copy()
df['target'] = df['breast_cancer_history'].astype(int)

print(f"\nAfter removing unknown outcomes: {df.shape}")
print(f"Total patients: {df['count'].sum():,}")
print(f"\nCancer prevalence:")
cancer_counts = df.groupby('target')['count'].sum()
total = cancer_counts.sum()
for label, cnt in cancer_counts.items():
    print(f"  {'No cancer' if label==0 else 'Cancer history'}: "
          f"{cnt:,} ({cnt/total*100:.1f}%)")

Loading BCSC data...
Raw shape: (1522340, 13)
Total patients represented: 6,788,436

After removing unknown outcomes: (1191438, 16)
Total patients: 5,712,811

Cancer prevalence:
  No cancer: 5,232,130 (91.6%)
  Cancer history: 480,681 (8.4%)


In [2]:
print("Expanding aggregated rows to individual patient records...")
print("(This may take 2-3 minutes for 5.7M patients)")

# select features available in BCSC
feature_cols_bcsc = [
    'age', 'first_degree_hx', 'age_menarche', 'age_first_birth',
    'BIRADS_breast_density', 'current_hrt', 'menopaus',
    'bmi', 'biophx'
]

# drop rows where ALL key features are missing
df_clean = df[feature_cols_bcsc + ['target', 'count']].copy()

# fill missing values with median per feature
for col in feature_cols_bcsc:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

# expand each row by its count
# instead of full expansion (too large), sample proportionally
# target ~100,000 individual records for training efficiency
np.random.seed(42)

total_patients = df_clean['count'].sum()
target_n = 150000
sample_prob = target_n / total_patients

rows = []
for _, row in df_clean.iterrows():
    n = row['count']
    # sample proportionally
    n_sample = max(1, round(n * sample_prob)) if n >= 5 else 0
    if n_sample > 0:
        for _ in range(n_sample):
            r = {col: row[col] for col in feature_cols_bcsc}
            r['target'] = row['target']
            rows.append(r)

df_expanded = pd.DataFrame(rows)

print(f"Expanded dataset: {len(df_expanded):,} records")
print(f"Cancer rate: {df_expanded['target'].mean()*100:.1f}%")
print(f"\nFeature summary:")
print(df_expanded[feature_cols_bcsc].describe().round(2).T[['mean','std','min','max']])

Expanding aggregated rows to individual patient records...
(This may take 2-3 minutes for 5.7M patients)
Expanded dataset: 244,737 records
Cancer rate: 7.5%

Feature summary:
                        mean    std   min   max
age                    53.12  11.84  21.0  82.0
first_degree_hx         0.17   0.38   0.0   1.0
age_menarche           12.04   0.67  11.0  14.0
age_first_birth        21.40   8.83   0.0  32.0
BIRADS_breast_density   2.41   0.75   1.0   4.0
current_hrt             0.06   0.23   0.0   1.0
menopaus                1.75   0.53   1.0   3.0
bmi                    22.06   4.57  17.0  33.0
biophx                  0.28   0.45   0.0   1.0


In [4]:
def engineer_bcsc_features(df):
    d = df.copy()
    
    # convert to int first to avoid float & int error
    d['current_hrt']  = d['current_hrt'].astype(int)
    d['menopaus']     = d['menopaus'].astype(int)
    d['BIRADS_breast_density'] = d['BIRADS_breast_density'].astype(int)
    d['age_menarche'] = d['age_menarche'].astype(int)
    d['age_first_birth'] = d['age_first_birth'].astype(int)

    # nulliparous flag
    d['nulliparous'] = (d['age_first_birth'] == 0).astype(int)

    # age x BMI interaction
    d['age_bmi_interaction'] = d['age'] * d['bmi'] / 100

    # postmenopausal flag
    d['postmenopausal'] = (d['menopaus'] >= 2).astype(int)

    # postmenopausal + HRT (highest risk combination)
    d['postmeno_hrt'] = ((d['postmenopausal'] == 1) & 
                          (d['current_hrt'] == 1)).astype(int)

    # early menarche flag
    d['early_menarche'] = (d['age_menarche'] <= 11).astype(int)

    # late first birth flag
    d['late_first_birth'] = (d['age_first_birth'] >= 30).astype(int)

    # dense breast flag (BIRADS 3 or 4)
    d['dense_breasts'] = (d['BIRADS_breast_density'] >= 3).astype(int)

    # age group
    d['age_group'] = pd.cut(d['age'],
        bins=[0, 40, 50, 60, 70, 100],
        labels=[0, 1, 2, 3, 4]).astype(int)

    return d

print("Engineering features...")
df_eng = engineer_bcsc_features(df_expanded)
print(f"Features engineered: {df_eng.shape}")

Engineering features...
Features engineered: (244737, 18)


In [5]:
feature_cols_final = [
    'age', 'first_degree_hx', 'age_menarche', 'age_first_birth',
    'BIRADS_breast_density', 'current_hrt', 'menopaus', 'bmi',
    'biophx', 'nulliparous', 'age_bmi_interaction', 'postmenopausal',
    'postmeno_hrt', 'early_menarche', 'late_first_birth',
    'dense_breasts', 'age_group'
]

X = df_eng[feature_cols_final].values
y = df_eng['target'].values

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train):,} ({y_train.sum():,} cancer, {y_train.mean()*100:.1f}%)")
print(f"Test:  {len(X_test):,}  ({y_test.sum():,} cancer, {y_test.mean()*100:.1f}%)")

# SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE: {len(X_train_sm):,} training samples (50/50 balanced)")

# scale
scaler_bcsc = StandardScaler()
X_train_scaled = scaler_bcsc.fit_transform(X_train_sm)
X_test_scaled  = scaler_bcsc.transform(X_test)

# train models
print("\nTraining models on real BCSC data...")
print(f"{'Model':<25} {'CV AUC':>10} {'Test AUC':>10} {'Test AP':>10}")
print("─" * 58)

models_bcsc = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4,
        learning_rate=0.05, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=4,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=42,
        eval_metric='logloss', verbosity=0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_bcsc = {}

for name, model in models_bcsc.items():
    cv_scores = cross_val_score(
        model, X_train_scaled, y_train_sm,
        cv=cv, scoring='roc_auc', n_jobs=-1)
    model.fit(X_train_scaled, y_train_sm)
    y_prob = model.predict_proba(X_test_scaled)[:,1]
    test_auc = roc_auc_score(y_test, y_prob)
    test_ap  = average_precision_score(y_test, y_prob)
    results_bcsc[name] = {
        'model': model, 'y_prob': y_prob,
        'test_auc': test_auc, 'test_ap': test_ap
    }
    print(f"  {name:<23} {cv_scores.mean():>9.4f} "
          f"{test_auc:>9.4f} {test_ap:>9.4f}")

# save best model
best_name = max(results_bcsc,
                key=lambda n: results_bcsc[n]['test_auc'])
print(f"\nBest model: {best_name} "
      f"(AUC={results_bcsc[best_name]['test_auc']:.4f})")

os.makedirs('../data/models', exist_ok=True)
joblib.dump(results_bcsc[best_name]['model'],
            '../data/models/mouna_bcsc_model.pkl')
joblib.dump(scaler_bcsc,
            '../data/models/scaler_bcsc.pkl')
joblib.dump(feature_cols_final,
            '../data/models/feature_cols_bcsc.pkl')
print("Models saved.")

Train: 195,789 (14,764.0 cancer, 7.5%)
Test:  48,948  (3,691.0 cancer, 7.5%)

After SMOTE: 362,050 training samples (50/50 balanced)

Training models on real BCSC data...
Model                         CV AUC   Test AUC    Test AP
──────────────────────────────────────────────────────────
  Logistic Regression        0.9156    0.9168    0.4028
  Gradient Boosting          0.9334    0.9256    0.4229
  XGBoost                    0.9330    0.9258    0.4229

Best model: XGBoost (AUC=0.9258)
Models saved.
